# Pump It Up — Multiclass Water Pump Classification

Portfolio version of an academic Machine Learning project based on the **Pump It Up – Data Mining the Water Table** competition.

The objective is to classify each water pump into one of three states:

- `functional`
- `functional needs repair`
- `non functional`

This public notebook focuses on the analytical workflow, model comparison and results. Competition datasets and row-level predictions are not redistributed.

## Key results

| Result | Value |
|---|---:|
| Best hold-out validation accuracy | **0.8107** |
| 3-fold stratified CV mean accuracy | **0.8042** |
| CV standard deviation | **0.0017** |
| Best public competition score | **0.8207** |
| Final documented submission score | **0.8183** |

The best public leaderboard score and the final documented notebook submission are kept separate because they correspond to different iterations of the project.

## Workflow

1. Initial data-quality assessment
2. Explicit missing-value analysis
3. Detection of masked zeros
4. Feature engineering
5. High-cardinality categorical handling
6. Stratified train/validation split
7. Random Forest, HistGradientBoosting and CatBoost comparison
8. CatBoost validation and cross-validation
9. Final training and competition submission

In [ ]:
from pathlib import Path
import pandas as pd

RESULTS_DIR = Path("../results")
DATA_DIR = Path("../data")

pd.read_csv(RESULTS_DIR / "model_comparison.csv")

## Data-quality decisions

A major part of the project was distinguishing explicit missing values from values that appeared to encode missing information as zero.

The original analysis treated zeros in `construction_year`, `gps_height`, `population` and `longitude` as suspicious masked missing values. Binary `*_was_zero` indicators were retained so the model could preserve information about the original encoding.

Additional engineered variables included temporal components from `date_recorded`, estimated `pump_age`, a combined regional variable and transformations for high-cardinality categorical features.

In [ ]:
# Public portfolio implementation
from pathlib import Path
import sys

sys.path.append(str(Path("..").resolve()))

from src.preprocessing import PumpPreprocessor
from src.modeling import build_catboost_model

preprocessor = PumpPreprocessor(top_n_categories=100)
model = build_catboost_model()

print(type(preprocessor).__name__)
print(type(model).__name__)

## Model comparison

CatBoost produced the strongest hold-out validation result among the three model families evaluated.

In [ ]:
comparison = pd.read_csv(RESULTS_DIR / "model_comparison.csv")
comparison.sort_values("validation_accuracy", ascending=False)

## CatBoost validation

The final validation analysis showed strong performance on the two majority classes, while the minority class `functional needs repair` remained substantially more difficult.

This is an important limitation: overall accuracy alone does not fully describe performance in an imbalanced multiclass problem.

In [ ]:
classification = pd.read_csv(RESULTS_DIR / "classification_report.csv")
classification

![CatBoost confusion matrix](../images/confusion_matrix_catboost.png)

The minority `functional needs repair` class achieved substantially lower recall than the other two classes, highlighting an area for future improvement.

## Feature importance

![CatBoost feature importance](../images/feature_importance_catboost.png)

The model's strongest signals include geographical, temporal and engineered features. Feature importance is descriptive of the fitted model and should not be interpreted causally.

## Competition result

The project was developed iteratively across multiple submissions. The best public score reached **0.8207**. The final notebook version documented a **0.8183** submission and was retained because it included the more complete data-quality, feature-engineering, model-comparison and validation workflow.

![Competition score](../images/competition_score.png)

## Main lessons

- Data cleaning materially affected the modeling strategy.
- Masked zeros required domain-aware treatment rather than blanket imputation.
- CatBoost was particularly suitable for mixed numerical/categorical inputs.
- Stratification and cross-validation helped assess stability.
- The minority repair class remained the main classification challenge.
- Competition score was treated as one signal, not a substitute for validation quality.

## Public portfolio boundary

This repository intentionally excludes:

- original competition datasets;
- row-level test predictions;
- submission CSV files;
- local/Colab-specific paths.

The public code is a cleaned portfolio refactor of the original academic workflow. Reported metrics come from the completed original project.